# Medical Disease Prediction: Data Cleaning, EDA & Model Training

This notebook implements a complete Machine Learning pipeline using patient symptom data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import joblib

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded successfully!')


Libraries loaded successfully!


## 1. Data Cleaning & Preprocessing

In [ ]:
train_df = pd.read_csv('data/Training.csv')
test_df = pd.read_csv('data/Testing.csv')
train_df = train_df.loc[:, ~train_df.columns.str.contains('^Unnamed')]
test_df = test_df.loc[:, ~test_df.columns.str.contains('^Unnamed')]
train_df = train_df.drop_duplicates().reset_index(drop=True)

target_col = 'prognosis'
X_train_raw = train_df.drop(columns=[target_col])
y_train_raw = train_df[target_col]
X_test_raw = test_df.drop(columns=[target_col])
y_test_raw = test_df[target_col]

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_test = label_encoder.transform(y_test_raw)
feature_names = list(X_train_raw.columns)
print(f"Total features: {len(feature_names)}, Total classes: {len(label_encoder.classes_)}")


Total features: 132, Total classes: 41


## 2. Model Training & Evaluation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, C=1.0, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=15, min_samples_split=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=42),
    'Naive Bayes': GaussianNB()
}

results = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    model.fit(X_train_raw, y_train)
    y_train_pred = model.predict(X_train_raw)
    y_test_pred = model.predict(X_test_raw)
    train_acc = float(accuracy_score(y_train, y_train_pred))
    test_acc = float(accuracy_score(y_test, y_test_pred))
    prec = float(precision_score(y_test, y_test_pred, average='weighted', zero_division=0))
    rec = float(recall_score(y_test, y_test_pred, average='weighted', zero_division=0))
    f1 = float(f1_score(y_test, y_test_pred, average='weighted', zero_division=0))
    cv_scores = cross_val_score(model, X_train_raw, y_train, cv=cv, scoring='accuracy')
    cv_mean = float(cv_scores.mean())
    gap = train_acc - test_acc
    status = 'Low / No Overfit' if gap <= 0.05 else ('Moderate Overfit' if gap <= 0.15 else 'High Overfit')
    results.append({'Model': name, 'Train Accuracy': round(train_acc, 4), 'Test Accuracy': round(test_acc, 4), 'Precision': round(prec, 4), 'Recall': round(rec, 4), 'F1-Score': round(f1, 4), 'CV Mean Accuracy': round(cv_mean, 4), 'Train-Test Gap': round(gap, 4), 'Status': status})

results_df = pd.DataFrame(results).sort_values(by='Test Accuracy', ascending=False)
print(results_df.to_string(index=False))


## 3. Save Model Artifacts

In [ ]:
joblib.dump(models['Logistic Regression'], 'saved_models/best_model.joblib')
joblib.dump(label_encoder, 'saved_models/label_encoder.joblib')
joblib.dump(feature_names, 'saved_models/feature_names.joblib')
results_df.to_json('saved_models/model_comparison.json', orient='records', indent=4)
print('Saved best_model.joblib cleanly!')


Saved best_model.joblib cleanly!
